<img src="https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Python_Programming/M2_D4_Vector.png"/>

# Employee Analytics at Vector 📊

You work as a data analyst at **Vector**, a company that helps organizations understand and optimize their workforce through data-driven insights.

The people operations team collects extensive data on employee performance, satisfaction, compensation, and work arrangements. They need you to apply statistical methods to understand workforce distributions, make reliable predictions, and draw valid conclusions from sample data rather than surveying the entire employee base.

The VP of People is preparing a board presentation on workforce analytics and needs statistically sound answers to questions about employee metrics. Your analysis will inform compensation decisions, remote work policies, and performance management strategies affecting 10,000+ employees.

## Your mission

You will build a comprehensive understanding of how employee metrics distribute across the organization. You will demonstrate that sample statistics reliably estimate population parameters, even with small samples. You will use this knowledge to make evidence-based recommendations for HR policies.


## Setup: Import libraries and configure visualization

Start by importing the statistical and visualization tools you need. Set consistent styling so all plots maintain professional appearance across your analysis.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
import pandas as pd

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_theme(style="whitegrid", palette="muted", context="notebook")

# Set random seed for reproducibility
np.random.seed(42)

## Task 1: Validate the 68-95-99.7 rule with employee heights

The health and wellness team wants to understand the distribution of employee heights for ergonomic workspace design. They heard about the empirical rule but want to see it validated with actual data.

**Context**: Employee heights follow a normal distribution with mean 170 cm and standard deviation 10 cm.

- Generate 10,000 employee heights from this distribution.
- Calculate what percentage of employees fall within 1, 2, and 3 standard deviations of the mean. 
- Compare your calculated percentages to the theoretical 68-95-99.7 rule. 
- Create a histogram with shaded regions showing these ranges. 
- Write your interpretation confirming whether the data matches the rule.

<Note type = "hint">

Use `np.random.normal(mean, std, size)` to generate heights. 

Calculate boundaries as mean ± k*std where k is 1, 2, or 3. Count values within each range using boolean indexing: `((heights >= lower) & (heights <= upper)).sum()`. 

For visualization, use `plt.axvspan()` to shade regions between boundaries with different alpha values for each standard deviation range.

</Note>

In [20]:
# Define population parameters
mu_height = 170  # mean height in cm
sigma_height = 10  # standard deviation in cm
n_employees = 10000

# Generate 10,000 heights
heights= np.random.normal(mu_height, sigma_height, n_employees)
# => N(mu=170, sigma=10)

# Calculate boundaries for 1, 2, and 3 standard deviations
bounds={
    1: (mu_height - sigma_height, mu_height + sigma_height),
    2: (mu_height - 2*sigma_height, mu_height + 2*sigma_height),
    3: (mu_height - 3*sigma_height, mu_height + 3*sigma_height)
}

# Count employees within each range
# Calculate percentages
counts = {}

percentages = {}

for k_empiric, (low, high) in bounds.items(): # low, high : valeurs extremes des intervalles ; k : clé du dictionnaire
    # logique .items : boucler sur le dictionnaire bounds
    mask = (heights >= low) & (heights <= high) # renvoie un tableau de booléens
    counts[k_empiric] = mask.sum() # nombre de True dans le tableau de booléens
    percentages[k_empiric] = counts[k_empiric] / n_employees * 100 # pourcentage
    print(f" {percentages[k_empiric]:.2f}% (empirical)")



# Print results comparing empirical to theoretical values
print("Empirical results:")
for k_theoric in [1,2,3]:
    # theoretical = [68, 95, 99.7][k_theoric-1]
    #  # index decalé de 1 dans la liste[index 0 = 68, index 1 = 95, index 2 = 99.7]
    # alternative plus lisible
    if k_theoric == 1:
        theoretical = 68
    elif k_theoric == 2:
        theoretical = 95
    else:
        theoretical = 99.7
   
    print(f"{k_theoric}σ : {percentages[k_theoric]:.2f}%  (theoretical ≈ {theoretical}%)")
    
    # attention ici je rappelle percentages ={68, 95, 99.7} => dictionnaire et je chope la valeur associée à la clé k_theoric
    # Le nom de la variable (k_empiric ou k_theoric) n'a aucune importance dans percentages, du moment qu'elle vaut 1, 2 ou 3, comme les clés du dictionnaire.
    
import plotly.graph_objects as go
# Create histogram with shaded regions
fig = go.Figure()

# Histogram
fig.add_trace(go.Histogram(
    x=heights,
    nbinsx=40,
    name="Heights",
    marker_color="steelblue",
    opacity=0.7
))

# Add shaded σ regions
colors = {
    1: "rgba(255,150,150,0.25)",
    2: "rgba(255,100,100,0.20)",
    3: "rgba(255,50,50,0.15)"
}

for k, (low, high) in bounds.items():
    fig.add_shape(
        type="rect",
        x0=low, x1=high,
        y0=0, y1=1,
        yref="paper",
        fillcolor=colors[k],
        line_width=0
    )

# Add vertical mean line
fig.add_shape(
    type="line",
    x0=mu_height, x1=mu_height,
    y0=0, y1=1,
    yref="paper",
    line=dict(color="black", width=2, dash="dash")
)

# --------------------------
# Layout
# --------------------------
fig.update_layout(
    title="Distribution des tailles avec zones 1σ, 2σ, 3σ (Plotly)",
    xaxis_title="Taille (cm)",
    yaxis_title="Effectif",
    bargap=0.05,
    template="plotly_white"
)

fig.show()

# YOUR INTERPRETATION HERE:
# Does your data confirm the 68-95-99.7 rule? What does this tell you about employee heights?
# Absolument, les données confirment la règle des 68-95-99.7. 
# Ce qui indique que la distribution des tailles des employés suit une distribution normale,
# où environ 68% des employés ont une taille dans un écart-type de la moyenne, 95% dans deux écarts-types, et 99.7% dans trois écarts-types.


 68.71% (empirical)
 95.60% (empirical)
 99.77% (empirical)
Empirical results:
1σ : 68.71%  (theoretical ≈ 68%)
2σ : 95.60%  (theoretical ≈ 95%)
3σ : 99.77%  (theoretical ≈ 99.7%)


## Task 2: Calculate probabilities for performance scores

The performance management team uses a standardized scoring system. Employees receive performance scores that distribute normally with mean 75 and standard deviation 12. Management wants to understand the probability ranges for different performance levels.

**Context**: Performance scores follow N(75, 12). Management defines "high performers" as employees scoring above 85.

- Calculate the probability that a randomly selected employee scores above 85. 
- Calculate the probability of scoring between 70 and 90. 
- Find the score threshold that separates the top 10% of performers. 
- Visualize these probabilities on a normal distribution curve with shaded regions.

<Note type = "hint">

Use `stats.norm.cdf(x, mean, std)` to find cumulative probability up to x. For probability above 85, calculate `1 - stats.norm.cdf(85, mu, sigma)`. For probability between two values, use `stats.norm.cdf(upper) - stats.norm.cdf(lower)`. To find the score for top 10%, use `stats.norm.ppf(0.90, mu, sigma)` where 0.90 represents the 90th percentile.

</Note>


In [21]:
# Define performance score parameters
mu_performance = 75
sigma_performance = 12

# Calculate probability of scoring above 85
p_above_85 = 1 - stats.norm.cdf(85, mu_performance, sigma_performance)
# => N(mu=75, sigma=12)

# Calculate probability of scoring between 70 and 90
p_70_90 = stats.norm.cdf(90, mu_performance, sigma_performance) - \
          stats.norm.cdf(70, mu_performance, sigma_performance)
          # P(70 < X < 90) = P(X < 90) - P(X < 70)
          # P(70 < X < 90)= 

# Find the score threshold for top 10% (90th percentile)
top10_threshold = stats.norm.ppf(0.90, mu_performance, sigma_performance)
# top10_threshold= stats.norm.cdf(0.10, mu_performance, sigma_performance)

# Print results with business interpretation
print(f"Probabilité score > 85 : {p_above_85:.3f}")
print(f"Probabilité score 70–90 : {p_70_90:.3f}")
print(f"Top 10% threshold : {top10_threshold:.2f}")

# Create visualization showing the normal distribution with shaded regions
x = np.linspace(40, 110, 500)
y = stats.norm.pdf(x, mu_performance, sigma_performance)
# Create figure
fig = go.Figure()

# Add line (equivalent of plt.plot)
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode="lines",
    line=dict(color="black"),
    name="Normal PDF"
))

fig.update_layout(
    title="Normal Distribution Curve (Plotly)",
    xaxis_title="Score",
    yaxis_title="Density",
    template="plotly_white"
)


# Shade area above 85, area between 70-90, and mark the 90th percentile

fig.add_trace(go.Scatter(
    x=x[x > 85],
    y=y[x > 85],
    mode="lines",
    fill="tozeroy",
    fillcolor="rgba(255,0,0,0.3)",
    line=dict(color="rgba(255,0,0,0)"),
    name="Score > 85"
))


mask_mid = (x > 70) & (x < 90)
fig.add_trace(go.Scatter(
    x=x[mask_mid],
    y=y[mask_mid],
    mode="lines",
    fill="tozeroy",
    fillcolor="rgba(0,255,0,0.3)",
    line=dict(color="rgba(0,255,0,0)"),
    name="70 < Score < 90"
))


fig.add_shape(
    type="line",
    x0=top10_threshold, x1=top10_threshold,
    y0=0, y1=max(y),
    line=dict(color="purple", width=2, dash="dash")
)

# --- 5) Add label for percentile ---
fig.add_annotation(
    x=top10_threshold,
    y=max(y)*0.9,
    text="90th percentile",
    showarrow=True,
    arrowhead=2,
    ax=20,
    ay=-30
)

# --- Layout ---
fig.update_layout(
    title="Distribution des performances avec zones probabilistes (Plotly)",
    xaxis_title="Score",
    yaxis_title="Density",
    template="plotly_white"
)

fig.show()

Probabilité score > 85 : 0.202
Probabilité score 70–90 : 0.556
Top 10% threshold : 90.38


## Task 3: Compare two employee populations

Vector acquired a competitor last year. HR wants to compare salary distributions between original Vector employees and acquired employees to ensure compensation equity across both groups.

**Context**: Original employees earn mean salary €65,000 with std €8,000. Acquired employees earn mean €62,000 with std €10,000. Both distributions are normal.

- Generate 1,000 salaries for each group. 
- Create overlapping histograms comparing both distributions. 
- Calculate the percentage of acquired employees earning less than the Vector median. 
- Determine what salary puts an acquired employee at the same percentile as a Vector employee earning €70,000.

<Note type = "hint">

Generate two separate arrays using `np.random.normal()` with different parameters. Find Vector median with `np.median()`. Calculate percentile of €70,000 in Vector distribution using `stats.norm.cdf(70000, mu_vector, sigma_vector)`. Then find equivalent salary in acquired distribution using `stats.norm.ppf(percentile, mu_acquired, sigma_acquired)`. For visualization, use `plt.hist()` twice with `alpha=0.6` for transparency and different colors.

</Note>

In [18]:
# Define salary parameters for both groups
mu_vector = 65000
sigma_vector = 8000
mu_acquired = 62000
sigma_acquired = 10000
n_sample = 1000

# Generate salaries for both groups
salaries_vector = np.random.normal(mu_vector, sigma_vector, n_sample)
salaries_acquired = np.random.normal(mu_acquired, sigma_acquired, n_sample)

# Calculate Vector median salary
median_vector = np.median(salaries_vector)

# Calculate percentage of acquired employees below Vector median
pct_below = (salaries_acquired < median_vector).mean()*100

# Find percentile of €70,000 in Vector distribution
p_70k = stats.norm.cdf(70000, mu_vector, sigma_vector)
# % des individus de la distribution Vector qui gagnent moins de 70k
# Find equivalent salary in acquired distribution at same percentile
equiv_salary = stats.norm.ppf(p_70k, mu_acquired, sigma_acquired)
# % des individus de la distribution Acquired qui ont le même score que 70k dans Vector

# Print comparison results
print(f"Pourcentage d'acquis gagnant < médiane Vector : {pct_below:.1f}%")
print(f"Salaire équivalent à 70k dans acquis : {equiv_salary:.0f}€")

# Create overlapping histograms
fig.add_trace(go.Histogram(
    x=salaries_vector,
    nbinsx=40,
    opacity=0.6,
    name="Vector",
    marker_color="blue"
))

# Histogram for Acquired
fig.add_trace(go.Histogram(
    x=salaries_acquired,
    nbinsx=40,
    opacity=0.6,
    name="Acquired",
    marker_color="orange"
))

# Overlay mode to mimic Matplotlib behavior
fig.update_layout(
    barmode='overlay',   # superpose les histogrammes (comme Matplotlib)
    title="Comparaison des distributions salariales",
    xaxis_title="Salaire (€)",
    yaxis_title="Effectif",
    template="plotly_white"
)

fig.show()

# YOUR INTERPRETATION HERE:
# Does this comparison suggest salary equity issues? What action should HR take?


Pourcentage d'acquis gagnant < médiane Vector : 61.9%
Salaire équivalent à 70k dans acquis : 68250€


## Task 4: Demonstrate the Central Limit Theorem with small samples

The survey team wants to understand if they can trust conclusions from small employee samples. They need proof that small samples produce reliable estimates of population parameters.

**Context**: True population satisfaction score is 7.2 out of 10 with standard deviation 1.8. The team conducts monthly surveys with only 30 employees.

- Simulate 10,000 surveys by repeatedly drawing samples of 30 employees and calculating each sample mean. 
- Create a histogram of these sample means. Compare the distribution of sample means to the theoretical sampling distribution predicted by the Central Limit Theorem. 
- Calculate what percentage of sample means fall within 0.3 points of the true population mean.

<Note type = "hint">

The Central Limit Theorem states that sample means distribute normally with mean = population mean and standard deviation = sigma/sqrt(n), called the standard error. 
Use a loop to generate 10,000 samples: `for i in range(10000): sample = np.random.normal(mu, sigma, sample_size); means.append(sample.mean())`. Calculate standard error as `sigma / np.sqrt(n)`. For theoretical curve, use `stats.norm.pdf(x, mu, standard_error)`. Count sample means within ±0.3 of mu using boolean indexing.

</Note>

In [5]:
# Define population parameters
mu_satisfaction = 7.2
sigma_satisfaction = 1.8
sample_size = 30
n_simulations = 10000

# Calculate theoretical standard error


# Generate 10,000 sample means
# Use a loop to draw samples and calculate their means


# Calculate mean and std of your sample means


# Calculate percentage within ±0.3 of true mean


# Print comparison of empirical vs theoretical values


# Create histogram with theoretical curve overlay
# Plot histogram of sample means, then overlay the theoretical normal curve


# YOUR INTERPRETATION HERE:
# Does CLT hold with n=30? Can the team trust their monthly surveys?


## Task 5: Explore how sample size affects precision

HR wants to optimize survey costs. They need to understand how sample size affects the reliability of their estimates. Larger samples cost more but provide more precise estimates.

**Context**: Population satisfaction is 7.2 with std 1.8. Compare sampling distributions for samples of size 10, 30, 100, and 300.

- For each sample size, generate 5,000 sample means. 
- Plot all four sampling distributions on one figure using subplots. 
- Calculate the standard error for each sample size. 
- Create a line plot showing how standard error decreases as sample size increases. 
- Recommend the minimum sample size needed to achieve standard error below 0.2.

<Note type = "hint">

Create a dictionary or list to store sample means for each sample size: `sample_sizes = [10, 30, 100, 300]`. For each size, repeat the sampling process from Task 4. Use `plt.subplots(2, 2)` to create a 2×2 grid of histograms. Standard error formula is `sigma / sqrt(n)`, so to find n for target SE: `n = (sigma / target_se) ** 2`. Plot standard errors against sample sizes using `plt.plot(sample_sizes, standard_errors, marker='o')`.

</Note>

In [6]:
# Define parameters
mu_satisfaction = 7.2
sigma_satisfaction = 1.8
sample_sizes = [10, 30, 100, 300]
n_simulations = 5000

# Generate sample means for each sample size
# Store results in a dictionary: {sample_size: [list of sample means]}


# Calculate standard error for each sample size


# Create 2×2 subplot showing histogram for each sample size
# Each subplot should show the distribution with its standard error noted


# Create line plot of standard error vs sample size
# Add horizontal line at SE = 0.2 as reference


# Calculate minimum sample size for SE < 0.2


# Print recommendation with cost-benefit reasoning


# YOUR INTERPRETATION HERE:
# How does sample size affect precision? What's the optimal balance between cost and accuracy?


## Task 6: Compare sampling distributions across different populations

Vector operates in three regions with different work cultures. Management wants to know if sample-based surveys work equally well across regions despite different population characteristics.

**Context**: Region A satisfaction (μ=7.5, σ=1.2), Region B (μ=6.8, σ=2.1), Region C (μ=7.0, σ=1.6). All use samples of n=40.

- Generate 3,000 sample means for each region. 
- Calculate the standard error for each region. 
- Create overlapping density plots showing all three sampling distributions. 
- Calculate the probability that a Region B sample mean exceeds 7.2. 
- Interpret whether CLT provides reliable estimates despite different population variances.

<Note type = "hint">

Store population parameters in dictionaries for clean code: `regions = {'A': {'mu': 7.5, 'sigma': 1.2}, ...}`. Generate sample means for each region separately. Standard error differs by region because SE = sigma/sqrt(n), and sigma varies. Use `sns.kdeplot()` or `plt.hist(density=True)` to create density plots that overlay nicely. For probability calculation, use `1 - stats.norm.cdf(7.2, mu_B, se_B)`.

</Note>

In [7]:
# Define regional parameters
regions = {
    'A': {'mu': 7.5, 'sigma': 1.2},
    'B': {'mu': 6.8, 'sigma': 2.1},
    'C': {'mu': 7.0, 'sigma': 1.6}
}
sample_size = 40
n_simulations = 3000

# Generate sample means for each region
# Store in dictionary: {region: [sample means]}


# Calculate standard error for each region


# Print standard errors comparison


# Create overlapping density plots for all three regions
# Use different colors and add legend


# Calculate probability that Region B sample mean exceeds 7.2


# Print probability with interpretation


# YOUR INTERPRETATION HERE:
# Does CLT work equally well across regions? Should HR adjust survey methodology by region?


## Task 7: Test CLT with non-normal population

The finance team questions whether CLT applies to salary data, which is right-skewed rather than normally distributed. They want proof that sample means still form a normal distribution even when the population does not.

**Context**: Employee salaries follow an exponential distribution (highly right-skewed) with mean €50,000. This represents a typical salary distribution where most employees earn moderate salaries but some earn much more.

- Generate a population of 100,000 salaries from an exponential distribution. 
- Plot the population histogram showing the skewed distribution. 
- Draw 5,000 samples of size 50 and calculate each sample mean. 
- Plot the sampling distribution of means. Compare the population distribution shape to the sampling distribution shape. 
- Calculate the standard error empirically and theoretically.

<Note type = "hint">

Generate exponential data using `np.random.exponential(scale=50000, size=100000)` where scale equals the mean. Exponential distributions have a special property: standard deviation equals the mean. Therefore, theoretical SE = mean/sqrt(n). Create side-by-side plots using `plt.subplots(1, 2)` to show population vs sampling distribution contrast. Use `np.std(sample_means)` for empirical SE and compare to `mu/np.sqrt(n)` for theoretical SE.

</Note>

In [8]:
# Define parameters
mu_salary = 50000
population_size = 100000
sample_size = 50
n_simulations = 5000

# Generate exponential population


# Calculate population mean and std


# Generate 5,000 sample means from this population
# Use np.random.choice() to draw samples from the population


# Calculate empirical standard error of sample means


# Calculate theoretical standard error
# For exponential: sigma = mu, so SE = mu/sqrt(n)


# Print comparison


# Create side-by-side plots
# Left: Population distribution (skewed), Right: Sampling distribution (normal)


# YOUR INTERPRETATION HERE:
# Does CLT transform a skewed distribution into normal? What's the minimum sample size needed?


## Task 8: Make business decisions using sampling distributions

Management sets a satisfaction score of 7.0 as the threshold for acceptable team morale. The VP wants to know: if next quarter's survey of 25 employees shows a mean of 6.8, should she be concerned? Or is this normal statistical variation?

**Context**: True population satisfaction is 7.2 with std 1.8. Surveys use samples of 25 employees. Threshold concern level is 7.0.

- Calculate the probability that a random sample of 25 shows a mean below 7.0. 
- Simulate 10,000 sample means to verify this probability empirically. 
- Visualize the sampling distribution with a marked threshold at 7.0. 
- Provide a clear business recommendation: should management worry if the next survey shows 6.8?

<Note type = "hint">

This is a practical application of normal probability. Calculate SE = sigma/sqrt(n). Find analytical probability using `stats.norm.cdf(7.0, mu, se)`. Simulate by generating samples and counting how many sample means fall below 7.0: `(sample_means < 7.0).sum() / len(sample_means)`. Create visualization with `plt.axvline(7.0)` to mark threshold. Shade the area below 7.0 using `plt.fill_between()`. Add text annotation showing the probability percentage.

</Note>

In [9]:
# Define parameters
mu_satisfaction = 7.2
sigma_satisfaction = 1.8
sample_size = 25
threshold = 7.0
n_simulations = 10000

# Calculate standard error


# Calculate analytical probability of sample mean < 7.0


# Simulate 10,000 sample means


# Calculate simulation probability


# Print comparison of analytical vs simulation


# Create visualization
# Plot theoretical distribution, mark threshold, shade area below threshold


# Print business interpretation
# Explain whether management should worry about a score of 6.8


# YOUR RECOMMENDATION HERE:
# What's your advice to the VP? At what score should she actually become concerned?
